# Detecção de Defeitos em PCBs utilizando o modelo YOLO  
## Desenvolvido por:  
>&nbsp;&nbsp;&nbsp;&nbsp;Alexandre Augusto Tescaro Oliveira  
>&nbsp;&nbsp;&nbsp;&nbsp;Felipe Dias Konda  
>&nbsp;&nbsp;&nbsp;&nbsp;Hugo Tahara Menegatti  

## 1. Introdução e Motivação

> **Resumo do Projeto:** O projeto consiste no desenvolvimento de um sistema de Visão Computacional para a detecção automática de defeitos em Placas de Circuito Impresso (PCBs), utilizando a arquitetura de Rede Neural Convolucional **YOLO11** (*You Only Look Once*). Foi feito um tratamento no dataset original para reduzir a quantidade de imagens para facilidar o treinamento do modelo (por motivos acadêmicos), onde reduzimos a quantidade de imagens de treinamento para ~4000 e de validação e teste para ~500. Isso foi feito de forma aleatória para manter uma boa distribuição de classes e evitar viéses, onde também aplicamos técnicas de Data Augmentation no pré-processamento dos dados. Foi possível obter um resultado excelente de métricas do modelo, onde é possível obter sucesso com praticamente todas as imagens do dataset e também com imagens de fontes externas (como as testadas no final do notebook).

### 1.1 Contextualização
No cenário da Indústria 4.0, a garantia de qualidade na fabricação de componentes eletrônicos é de extrema importância para evitar problemas operacionais futuros. As Placas de Circuito Impresso (PCBs) são a base de praticamente todos os dispositivos eletrônicos modernos. Com a crescente miniaturização dos componentes, a complexidade dessas placas aumentou exponencialmente, tornando o processo de controle de qualidade cada vez mais desafiador.

### 1.2 O Problema
Tradicionalmente, a inspeção de PCBs é realizada de forma manual por operadores humanos ou por algoritmos de visão clássica baseados em regras rígidas. Esses métodos apresentam limitações significativas:
>* **Fadiga Humana:** A inspeção visual repetitiva leva à fadiga, resultando em erros de detecção e inconsistência.
>* **Baixa Escalabilidade:** A inspeção manual é lenta e cria gargalos na linha de produção.
>* **Limitações da Visão Clássica:** Algoritmos tradicionais muitas vezes falham em lidar com variações de iluminação, rotação ou ruídos na imagem.

### 1.3 A Solução Proposta
Para reduzir esses problemas e automatizar o trabalho, este projeto propõe o uso de Deep Learning. Utilizamos o modelo YOLO11, uma arquitetura de single-stage detector conhecida por seu equilíbrio entre alta precisão e velocidade de inferência em tempo real.

O objetivo é identificar e localizar (através de Bounding Boxes) seis tipos comuns de defeitos de fabricação:
>1.  **Missing Hole** (Furo faltante)
>2.  **Mouse Bite** (Mordida de rato/Falha na borda)
>3.  **Open Circuit** (Circuito aberto)
>4.  **Short** (Curto-circuito)
>5.  **Spur** (Esporão/Rebarba)
>6.  **Spurious Copper** (Cobre residual)

A implementação deste modelo visa aumentar a eficiência do controle de qualidade, reduzindo o desperdício de material e o escape de produtos defeituosos para o mercado.

## 2. Análise Exploratória dos Dados (EDA)  
> Link de acesso ao notebook realizado: https://colab.research.google.com/drive/1Zy-WgUTB66TsTryg1J9_lR-0EY2i_zgV?usp=sharing  
> Link de acesso ao Dataset utilizado: https://www.kaggle.com/datasets/norbertelter/pcb-defect-dataset

In [11]:
pip install pyyaml torch pandas matplotlib seaborn ultralytics Pillow numpy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import yaml
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
from PIL import Image
import numpy as np
import glob

In [4]:
# ==============================================================================
# CONFIGURAÇÃO DOS CAMINHOS DO DATASET
# ==============================================================================

# Caminho base do diretório do dataset
BASE_DIR = "pcb-defect-dataset"
print(f"Diretório base do Notebook: {BASE_DIR}")

# Caminhos para as imagens
train_images_dir = os.path.join(BASE_DIR, 'train', 'images')
val_images_dir = os.path.join(BASE_DIR, 'val', 'images')
test_images_dir = os.path.join(BASE_DIR, 'test', 'images')

# Verificação de segurança
if not os.path.exists(train_images_dir):
    print(f"ERRO: Não encontrei a pasta de treino em: {train_images_dir}")
    print("Certifique-se de que este notebook está salvo na pasta RAIZ do seu dataset.")
else:
    print("Pastas de treino encontradas!")

    # Cria o dicionário de configuração do YOLO
    data_yaml = {
        'path': BASE_DIR,           # Raiz do dataset
        'train': 'train/images',    # Caminho relativo para treino
        'val': 'val/images',        # Caminho relativo para validação
        'test': 'test/images',      # Caminho relativo para teste

        # Nomes das classes
        'names': {
            0: 'mouse_bite',
            1: 'spur',
            2: 'missing_hole',
            3: 'short',
            4: 'open_circuit',
            5: 'spurious_copper'
        }
    }

    yaml_path =  os.path.join("pcb-defect-dataset", "data.yaml")
    with open(yaml_path, 'w') as f:
        yaml.dump(data_yaml, f)

    print(f"Arquivo de configuração criado em: {yaml_path}")

Diretório base do Notebook: pcb-defect-dataset
Pastas de treino encontradas!
Arquivo de configuração criado em: pcb-defect-dataset/data.yaml


In [14]:
# ==============================================================================
# CHECAR DATA LEAKS
# ==============================================================================

def check_filename_leakage(train_dir, val_dir):
    # Pega apenas os nomes dos arquivos
    train_files = set(os.listdir(train_dir))
    val_files = set(os.listdir(val_dir))

    print(f"Total Treino: {len(train_files)}")
    print(f"Total Validação: {len(val_files)}")

    # Checa duplicatas exatas de nome
    duplicates = train_files.intersection(val_files)
    if duplicates:
        print(f"PERIGO: {len(duplicates)} arquivos têm EXATAMENTE o mesmo nome em Treino e Validação!")
        print(list(duplicates)[:5])
    else:
        print("Nomes de arquivos exatos não se repetem.")

    #  Checa vazamento por Prefixo (Assumindo que o prefixo indica a placa de origem)
    train_prefixes = set([f.split('_')[0] for f in train_files])
    val_prefixes = set([f.split('_')[0] for f in val_files])

    prefix_leak = train_prefixes.intersection(val_prefixes)

    if prefix_leak:
        print(f"ATENÇÃO: {len(prefix_leak)} placas originais (prefixos) aparecem em AMBOS os conjuntos.")
        print(f"Exemplos: {list(prefix_leak)[:5]}")
    else:
        print("Prefixos distintos. Parece que as placas foram separadas corretamente.")


if os.path.exists(train_images_dir) and os.path.exists(val_images_dir):
    check_filename_leakage(train_images_dir, val_images_dir)

Total Treino: 8534
Total Validação: 1066
Nomes de arquivos exatos não se repetem.
ATENÇÃO: 3 placas originais (prefixos) aparecem em AMBOS os conjuntos.
Exemplos: ['l', 'rotation', 'light']


## 3. Arquitetura do Modelo: YOLO11

Neste projeto, utilizamos o modelo de detecção de objetos: **YOLO11 (You Only Look Once)**.

O YOLO se diferencia por "olhar" para a imagem apenas uma vez, dividindo-a em uma grade e prevendo simultaneamente onde estão os objetos e o que eles são.

### Por que YOLO para PCBs?
A escolha desta arquitetura baseia-se em três fatores importantes para inspeção de qualidade:

>* **Velocidade:** Por ser um detector de estágio único (*single-stage*), permite a verificação em milissegundos, viabilizando o uso em esteiras de produção rápidas.
>* **Detecção Multiescala:** Graças ao seu componente **Neck**, o modelo consegue "misturar" características de alta e baixa resolução. Isso é fundamental para nosso dataset, que contém defeitos muito pequenos (ex: *furos faltantes*) e maiores (ex: *curtos-circuitos*).
>* **Anchor-Free:** O YOLO não depende de "moldes" fixos de caixas. Ele aprende a geometria exata do defeito, adaptando-se melhor a falhas irregulares como spurs ou mouse bites.

### Estrutura Simplificada
O fluxo de dados dentro do modelo segue estas etapas:

>1.  **Input:** Imagem da PCB (redimensionada para 640x640).
>2.  **Backbone (CSPDarknet):** Extrai as características visuais (Feature Extractor), através de camadas convolucionais com Kernel 2x2.
>3.  **Neck (PANet):** Combina detalhes finos com o contexto global, através da função de ativação SiLu.
>4.  **Head:** Gera as saídas finais:
>    * *Coordenadas do Bounding Box:* `[x, y, largura, altura]`
>    * *Classe:* `[probabilidade do defeito]`  

<div align="center">
  <h3>Arquitetura YOLO</h3>
  <img src="https://www.researchgate.net/publication/329038564/figure/fig2/AS:694681084112900@1542636285619/YOLO-architecture-YOLO-architecture-is-inspired-by-GooLeNet-model-for-image.ppm" width="700" alt="Diagrama YOLO">
  <p><em>Figura 1: Esquema do Backbone, Neck e Head do YOLO.</em></p>
</div>

A imagem acima representa a arquitetura inicial do YOLO. Versões mais modernas não possuem mais as camadas Fully Conected no final, se tratanto de um modelo com uma arquitetura totalmente convolucional.Ela segue as seguintes convoluções até a saída final:  
### 1. Entrada
* A imagem entra e sofre convoluções para reduzir o seu tamanho.
* **Efeito:** A imagem é reduzida rapidamente (*Downsample*) para 1/4 do tamanho original, transformando pixels brutos em características básicas.

### 2. Backbone (Extração com C3k2)
* A imagem passa por vários blocos **C3k2**, aplicando convoluções de tamanho variável.
* A cada estágio, uma Convolução de Kernel 2x2 reduz o tamanho da imagem pela metade, enquanto dobra o número de canais (profundidade).
* No final, o bloco **C2PSA** aplica mecanismos de atenção para destacar as regiões de interesse (os defeitos).

### 3. Neck (Fusão com PANet)
* O modelo utiliza **Upsample** (aumenta a resolução da imagem) e **Convoluções $1 \times 1$**.
* **Objetivo:** Misturar as características profundas (semânticas/abstratas) com as características rasas (detalhadas/visuais).

### 4. Head (Predição)
* Aplica **Convoluções $1 \times 1$** finais para gerar os vetores de saída independentes:
    1.  Um vetor para a caixa (**Regressão**).
    2.  Um vetor para a classe (**Classificação**).

In [15]:
# ==============================================================================
# TREINAMENTO DO MODELO
# ==============================================================================

if torch.cuda.is_available():
    print(f"GPU NVIDIA Detectada: {torch.cuda.get_device_name(0)}")
    device_id = 0 # Usa a GPU 0
elif torch.backends.mps.is_available():
    print("🚀 Apple Silicon GPU (MPS) detectada! O treinamento será acelerado no seu M4.")
    device_id = 'mps' # Usa a GPU do Mac
else:
    print("GPU não detectada. O treinamento será feito na CPU.")
    device_id = 'cpu'

# Carrega o modelo pré-treinado (transfer learning)
model = YOLO('yolo11n.pt')
print("🚀 Iniciando o treinamento...")


# utilizar workers=0 no windows para evitar problemas de desempenho
results = model.train(
    data=yaml_path, # Certifique-se de que a variável yaml_path está definida

    # HIPERPARÂMETROS
    epochs=50,
    patience=10,
    imgsz=640,
    batch=16,            # Reduzido de 32 para 16 (veja a explicação abaixo)
    project='resultado_projeto',
    name='treino_robustez_v2',
    workers=4,           # No macOS você pode e deve usar mais workers (tente 4 ou 8)
    lr0=0.001,
    device=device_id,    # Aqui ele vai passar 'mps'

    # REGULARIZAÇÃO
    optimizer='AdamW',
    weight_decay=0.0005,
    dropout=0.1,

    # DATA AUGMENTATION FOTOMÉTRICO
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    # DATA AUGMENTATION GEOMÉTRICO
    augment=True,
    degrees=15.0,
    perspective=0.0005,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.5,

    # MIXAGEM DE CONTEXTO
    mosaic=1.0,
    mixup=0.1
)

print(f"✅ Treinamento concluído! Resultados salvos em: {results.save_dir}")

🚀 Apple Silicon GPU (MPS) detectada! O treinamento será acelerado no seu M4.
🚀 Iniciando o treinamento...
Ultralytics 8.4.21 🚀 Python-3.14.3 torch-2.10.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=pcb-defect-dataset/data.yaml, degrees=15.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=treino_robustez_v2, nbs=64, n

## 4. Análise de Métricas de Desempenho

Para validar a eficácia do modelo YOLO na detecção de defeitos em Placas de Circuito Impresso (PCBs), utilizamos um conjunto robusto de métricas de avaliação. Abaixo, detalhamos cada métrica e sua relevância no contexto de inspeção industrial.

## Matriz de Confusão
A **Matriz de Confusão** é a ferramenta fundamental para visualizar os erros do modelo. Ela compara, classe por classe, a previsão do modelo versus a realidade (Ground Truth).

* **Definição Técnica:** Uma tabela onde as linhas representam as classes reais e as colunas representam as classes preditas. A diagonal principal indica os acertos.
* **Contexto do Projeto:** Em nossa análise, a matriz permite identificar "confusões funcionais".
    * *Exemplo Crítico:* Se o modelo confundir `mouse_bite` (mordida) com `open_circuit` (circuito aberto), o erro é menos grave, pois ambos indicam falha de continuidade.
    * *Exemplo Grave:* Se o modelo classificar um defeito `short` (curto) como `background` (fundo), temos um **falso negativo**, o que significa que uma placa defeituosa seria enviada ao cliente.

## Precisão (Precision)
A precisão responde à pergunta: **"De todos os defeitos que o modelo apontou, quantos eram reais?"**

$$\text{Precision} = \frac{\text{Verdadeiros Positivos}}{\text{Verdadeiros Positivos} + \text{Falsos Positivos}}$$

* **No nosso Projeto:** Uma alta precisão significa que o modelo gera poucos **alarmes falsos**.
* **Impacto Industrial:** Se a precisão for baixa, a linha de produção irá descartar muitas placas boas (desperdício de material e dinheiro), pois o modelo estará "alucinando" defeitos onde não existem.

## Recall (Sensibilidade)
O Recall responde à pergunta: **"De todos os defeitos que existiam na placa, quantos o modelo conseguiu encontrar?"**

$$\text{Recall} = \frac{\text{Verdadeiros Positivos}}{\text{Verdadeiros Positivos} + \text{Falsos Negativos}}$$

* **No nosso Projeto:** Esta é, possivelmente, a métrica mais crítica para controle de qualidade.
* **Impacto Industrial:** Um baixo Recall significa que o modelo está deixando passar defeitos ("escapes"). Isso é perigoso, pois resulta no envio de produtos defeituosos para o mercado, causando falhas em dispositivos eletrônicos e danos à reputação da fábrica.
* **Nossos Resultados:** O modelo obteve Recall próximo de **100%** para a classe crítica `missing_hole`, garantindo que nenhuma placa sem furos prossiga na linha.

## F1-Score
O F1-Score é a média harmônica entre Precisão e Recall. Ele resume a qualidade do modelo em um único número, penalizando modelos desequilibrados (ex: que acham tudo mas erram muito, ou que são precisos mas não acham nada).

$$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

* **Contexto do Projeto:** Buscamos um F1-Score alto (> 0.90) para garantir que o sistema seja **confiável** (não minta) e **seguro** (não deixe passar falhas). Nosso F1 de **0.99** indica um equilíbrio quase perfeito.

## mAP (Mean Average Precision)
Esta é a métrica padrão-ouro para Detecção de Objetos.

* **mAP@50:** Calcula a média da precisão considerando um acerto qualquer caixa que tenha pelo menos **50% de sobreposição (IoU)** com o defeito real.
    * *Interpretação:* Indica o quão bom o modelo é em **localizar** e **classificar** o defeito corretamente.
* **mAP@50-95:** É uma métrica mais rigorosa que faz a média de vários limiares (50% a 95%).
    * *Interpretação:* Indica o quão "perfeita" e ajustada é a caixa desenhada.
* **Contexto do Projeto:** Para fins de inspeção, o **mAP@50** é o indicador mais relevante. Se o modelo detectar que há um `short` em uma determinada área (mesmo que a caixa seja um pouco maior que o defeito), a placa já será corretamente reprovada.

## Funções de Perda (Loss Functions)
Durante o treinamento, monitoramos três tipos de "erro" que o modelo tenta minimizar:

1.  **Box Loss (Erro de Caixa):** O quão longe a caixa prevista está da caixa real. Mede o erro de coordenadas $(x, y, w, h)$.
2.  **Cls Loss (Erro de Classe):** O quão errado o modelo estava sobre o tipo de defeito (ex: dizer que é `spur` quando era `short`).
3.  **DFL Loss (Distribution Focal Loss):** Uma métrica auxiliar usada pelo YOLO para refinar a precisão das bordas da caixa.

**Análise das Curvas:** A convergência simultânea dessas perdas (descida constante e estabilização) sem o aumento da perda de validação confirmou que o modelo aprendeu de forma saudável, sem sofrer de *overfitting* (decora) ou *underfitting* (incapacidade de aprender).

In [16]:
print("Rodando validação final...")
metrics = model.val(split='test')

# Métricas Padrão
print(f"\n--- 📊 Métricas de Desempenho ---")
print(f"mAP@50 (Acurácia de Detecção): {metrics.box.map50:.4f}")
print(f"mAP@50-95 (Precisão Geral):    {metrics.box.map:.4f}")
print(f"Precisão (Precision):          {metrics.box.mp:.4f}")
print(f"Recall (Sensibilidade):        {metrics.box.mr:.4f}")

# Cálculo do F1-Score
f1_scores = metrics.box.f1
if hasattr(f1_scores, 'ndim') and f1_scores.ndim == 1:
    f1_max = f1_scores.max()
else:
    f1_max = np.mean(f1_scores[:, np.argmax(f1_scores.mean(0))])
print(f"F1-Score Máximo:               {f1_max:.4f}")

# Gráficos de Aprendizado
try:
    run_dir = model.trainer.save_dir
except AttributeError:
    # Fallback: se results for lista, tenta pegar do primeiro item
    if isinstance(results, list):
        run_dir = results[0].save_dir
    else:
        run_dir = results.save_dir

print(f"Diretório dos resultados: {run_dir}")

csv_path = os.path.join(run_dir, 'results.csv')

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    plt.figure(figsize=(18, 5))

    # Gráfico 1: Curvas de Perda
    plt.subplot(1, 3, 1)
    train_loss = df['train/box_loss'] + df['train/cls_loss'] + df['train/dfl_loss']
    # Verifica se existe validação no CSV antes de somar
    if 'val/box_loss' in df.columns:
        val_loss = df['val/box_loss'] + df['val/cls_loss'] + df['val/dfl_loss']
        plt.plot(df['epoch'], val_loss, label='Validação', color='orange', linestyle='--', linewidth=2)
    plt.plot(df['epoch'], train_loss, label='Treino', color='red', linewidth=2)
    plt.title('Curva de Perda (Loss) Global')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Gráfico 2: Acurácia (mAP)
    map50_col = 'metrics/mAP50(B)' if 'metrics/mAP50(B)' in df.columns else 'metrics/mAP50'
    map95_col = 'metrics/mAP50-95(B)' if 'metrics/mAP50-95(B)' in df.columns else 'metrics/mAP50-95'

    plt.subplot(1, 3, 2)
    plt.plot(df['epoch'], df[map50_col], label='mAP@50', color='blue', linewidth=2)
    plt.plot(df['epoch'], df[map95_col], label='mAP@50-95', color='green', linestyle='--')
    plt.title('Acurácia (mAP)')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Gráfico 3: Precisão vs Recall
    p_col = 'metrics/precision(B)' if 'metrics/precision(B)' in df.columns else 'metrics/precision'
    r_col = 'metrics/recall(B)' if 'metrics/recall(B)' in df.columns else 'metrics/recall'

    plt.subplot(1, 3, 3)
    plt.plot(df['epoch'], df[p_col], label='Precision', color='purple')
    plt.plot(df['epoch'], df[r_col], label='Recall', color='cyan')
    plt.title('Precisão vs Recall')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Arquivo results.csv não encontrado.")

# 3. Exibir Matriz de Confusão
cm_paths = [
    os.path.join(run_dir, 'confusion_matrix_normalized.png'),
    os.path.join(run_dir, 'confusion_matrix.png')
]

found_cm = False
for p in cm_paths:
    if os.path.exists(p):
        plt.figure(figsize=(8, 8))
        plt.imshow(Image.open(p))
        plt.axis('off')
        plt.title("Matriz de Confusão")
        plt.show()
        found_cm = True
        break

if not found_cm:
    print("⚠️ Imagem da Matriz de Confusão não encontrada.")

Rodando validação final...
Ultralytics 8.4.21 🚀 Python-3.14.3 torch-2.10.0 CPU (Apple M4)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 210.6±96.8 MB/s, size: 86.7 KB)
val: Scanning /Users/alehholiveira/Desktop/Projetos/neural-links-pcb-defect-detection/pcb-defect-dataset/test/labels... 829 images, 239 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1068/1068 7.4Kit/s 0.1s.1s
val: New cache created: /Users/alehholiveira/Desktop/Projetos/neural-links-pcb-defect-detection/pcb-defect-dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 2.1s/it 2:212.3ss
                   all       1068       1662      0.979      0.973      0.986      0.546
            mouse_bite        131        262      0.981      0.979      0.982       0.51
                  spur        138        279      0.986      0.957      0.985      0.535
        

<Figure size 1800x500 with 3 Axes>

<Figure size 800x800 with 1 Axes>

## 5. Testes de Inferência

In [5]:
import matplotlib.pyplot as plt
from PIL import Image
import os
from IPython.display import display # Força a exibição no Jupyter

# Comando para garantir que a imagem irá aparecer na célula
%matplotlib inline

# Chamada do melhor modelo treinado anteriormente
model_path = os.path.join(BASE_DIR, 'resultado_projeto', 'treino_robustez_v2', 'weights', 'best.pt')
best_model = YOLO(model_path)

# Caminho e imagem escolhida para testar o modelo
imagem_escolhida = os.path.join(test_images_dir, "rotation_90_light_12_short_03_4_600.jpg")
print(f"🔍 Analisando: {imagem_escolhida}")

# Predição
results = best_model.predict(imagem_escolhida, conf=0.25)

# Visualização e Salvamento
for i, result in enumerate(results):
    im_array = result.plot()
    im = Image.fromarray(im_array[..., ::-1]) # BGR -> RGB

    # B. SALVA NO DISCO (Garantia 100%)
    save_path = os.path.join(BASE_DIR, f"resultado_teste_{i}.png")
    im.save(save_path)
    print(f"✅ Imagem com detecções salva em: {save_path}")

    # C. MOSTRA NA TELA (Tentativa robusta)
    print("Tentando exibir abaixo...")
    plt.figure(figsize=(12, 12))
    plt.imshow(im)
    plt.axis('off')
    plt.title(f"Detectado: {len(result.boxes)} defeitos")
    plt.show() # Comando Matplotlib

    # D. Exibição redundante (caso o plt falhe)
    display(im)

FileNotFoundError: [Errno 2] No such file or directory: 'pcb-defect-dataset/resultado_projeto/treino_robustez_v2/weights/best.pt'

In [ ]:
%matplotlib inline


model_path = os.path.join('', 'resultado_projeto', 'treino_robustez_v2', 'weights', 'best.pt')
best_model = YOLO(model_path)


imagem_escolhida = os.path.join(test_images_dir, "l_light_06_mouse_bite_02_5_600.jpg")

print(f"🔍 Analisando: {imagem_escolhida}")

results = best_model.predict(imagem_escolhida, conf=0.25)


for i, result in enumerate(results):
    im_array = result.plot()
    im = Image.fromarray(im_array[..., ::-1]) # BGR -> RGB


    save_path = os.path.join("", f"resultado_teste_{i}.png")
    im.save(save_path)
    print(f"✅ Imagem com detecções salva em: {save_path}")


    print("Tentando exibir abaixo...")
    plt.figure(figsize=(12, 12))
    plt.imshow(im)
    plt.axis('off')
    plt.title(f"Detectado: {len(result.boxes)} defeitos")
    plt.show()


    display(im)

In [ ]:

%matplotlib inline


model_path = os.path.join('', 'resultado_projeto', 'treino_robustez_v2', 'weights', 'best.pt')
best_model = YOLO(model_path)

imagem_escolhida = os.path.join("", "imagem_exemplo.png")

print(f"🔍 Analisando: {imagem_escolhida}")


results = best_model.predict(imagem_escolhida, conf=0.25)

for i, result in enumerate(results):

    im_array = result.plot()
    im = Image.fromarray(im_array[..., ::-1]) # BGR -> RGB


    save_path = os.path.join("", f"resultado_teste_{i}.png")
    im.save(save_path)
    print(f"✅ Imagem com detecções salva em: {save_path}")


    print("Tentando exibir abaixo...")
    plt.figure(figsize=(12, 12))
    plt.imshow(im)
    plt.axis('off')
    plt.title(f"Detectado: {len(result.boxes)} defeitos")
    plt.show()


    display(im)

In [ ]:
%matplotlib inline

model_path = os.path.join('', 'resultado_projeto', 'treino_robustez_v2', 'weights', 'best.pt')
best_model = YOLO(model_path)


imagem_escolhida = os.path.join("", "image2.png")

print(f"🔍 Analisando: {imagem_escolhida}")


results = best_model.predict(imagem_escolhida, conf=0.25)


for i, result in enumerate(results):

    im_array = result.plot()
    im = Image.fromarray(im_array[..., ::-1]) # BGR -> RGB


    save_path = os.path.join("", f"resultado_teste_{i}.png")
    im.save(save_path)
    print(f"✅ Imagem com detecções salva em: {save_path}")


    print("Tentando exibir abaixo...")
    plt.figure(figsize=(12, 12))
    plt.imshow(im)
    plt.axis('off')
    plt.title(f"Detectado: {len(result.boxes)} defeitos")
    plt.show()


    display(im)

## 6. Conclusão  

Neste projeto, conseguimos comprovar que o uso de Inteligência Artificial, especificamente a arquitetura **YOLO**, funciona muito bem para encontrar defeitos em placas eletrônicas (PCBs).

Os testes mostraram que o nosso modelo não só aprendeu, como atingiu um desempenho excelente. Os pontos principais foram:

* **Precisão:** O modelo alcançou um **mAP@50 de 98.1%** e um **F1-Score de 0.98**. Na prática, isso significa que ele é muito confiável: raramente deixa passar um defeito e quase nunca da um falso positivo.
* **Velocidade:** Ele processa cada imagem em cerca de **5 milissegundos**, um tempo muito eficiente para ser aplicado em um ambiente real. Isso resolveria o problema de inspeção humana, que teria um tempo de reação de no mínimo **300ms**.
* **Aprendizado Real:** Graças às técnicas de Data Augmentation e aos ajustes que fizemos (como o AdamW e técnica L2), o modelo aprendeu a identificar os padrões dos defeitos de verdade, em vez de apenas "decorar" as imagens de treino.

Na análise dos erros, notamos que o modelo só teve um pouco de dificuldade em casos muito específicos, como diferenciar uma "mordida" na placa (*mouse_bite*) de um circuito aberto, o que é compreensível pois visualmente são falhas bem parecidas.

### Próximos Passos

Embora os resultados no computador tenham sido ótimos, sabemos que levar isso para uma fábrica real traz novos desafios. Para continuar evoluindo o projeto, pensamos nas seguintes etapas:

### Testar com Imagens Reais
O dataset que usamos é muito padronizado. O próximo passo ideal seria tirar fotos reais de placas com câmeras comuns ou celulares, com iluminação variada e ângulos tortos, para ver se o modelo mantém esse desempenho todo.

### Rodar em Computadores Simples
Para não depender de servidores caros e ser mais aplicável em um ambiente real, seria interessante otimizar o modelo para rodar em dispositivos pequenos e baratos (como por exemplo com câmeras ruins), como um **Raspberry Pi**, que poderiam ser instalados diretamente na linha de produção.

### Criar uma Interface para o Usuário
A ideia seria criar um site ou aplicativo simples onde o operador da fábrica pudesse ver as detecções na tela e, se necessário, corrigir o modelo caso ele erre, ajudando a IA a ficar cada vez mais inteligente.

# 7. Referências  
> Documentação YOLO: https://docs.ultralytics.com/pt/  
> Documentação Pytorch: https://docs.pytorch.org/docs/stable/index.html  
> Documentação CSP-Net: https://huggingface.co/docs/timm/models/csp-darknet  
> Stanford CNN Cheatsheet: https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-convolutional-neural-networks  
> Materiais de Aula  